In [4]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless

!pip install opencv-contrib-python-headless numpy requests

Found existing installation: opencv-contrib-python-headless 5.0.0.93
Uninstalling opencv-contrib-python-headless-5.0.0.93:
  Successfully uninstalled opencv-contrib-python-headless-5.0.0.93
  Using cached opencv_contrib_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_contrib_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (69.5 MB)


In [1]:
import cv2
import numpy as np

print("OpenCV Version:", cv2.__version__)
print("HOG Available:", hasattr(cv2, "HOGDescriptor"))

OpenCV Version: 5.0.0
HOG Available: True


In [2]:
from google.colab import files

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

print("Uploaded video:", video_path)

Saving video.mp4 to video (1).mp4
Uploaded video: video (1).mp4


In [3]:
hog = cv2.HOGDescriptor()

hog.setSVMDetector(
    cv2.HOGDescriptor_getDefaultPeopleDetector()
)

print("Pedestrian Detection Model Loaded Successfully!")

Pedestrian Detection Model Loaded Successfully!


In [4]:
import requests
import time
from getpass import getpass

token = getpass("Enter your NEW Pushbullet API token (or press Enter to skip): ")

print("Pushbullet configured." if token else "Pushbullet notifications disabled.")

Enter your NEW Pushbullet API token (or press Enter to skip): ··········
Pushbullet notifications disabled.


In [5]:
def estimate_distance(height):
    focal_length = 500
    real_height = 1.7

    return (focal_length * real_height) / height


def classify_movement(prev_y, current_y):
    speed = abs(current_y - prev_y)

    return "Standing" if speed < 5 else "Walking"


def send_notification(distance):

    if not token:
        return

    try:
        response = requests.post(
            "https://api.pushbullet.com/v2/pushes",
            headers={
                "Access-Token": token,
                "Content-Type": "application/json"
            },
            json={
                "type": "note",
                "title": "Pedestrian Alert!",
                "body": f"Pedestrian detected at {distance:.1f} meters!"
            },
            timeout=10
        )

        print("Pushbullet status:", response.status_code)

    except Exception as e:
        print("Notification error:", e)


print("Functions loaded successfully!")

Functions loaded successfully!


In [6]:
from IPython.display import Video, display

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Cannot open uploaded video!")

fps = cap.get(cv2.CAP_PROP_FPS)

if not fps or fps <= 0:
    fps = 25

width = 480
height = 360

output_path = "pedestrian_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

prev_positions = {}
last_push_time = 0
push_cooldown = 10
frame_count = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.resize(frame, (width, height))

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    rects, weights = hog.detectMultiScale(
        gray,
        winStride=(8, 8),
        padding=(8, 8),
        scale=1.05
    )

    for i, (x, y, w, h) in enumerate(rects):

        distance = estimate_distance(h)

        if distance < 5:

            color = (0, 0, 255)

            if time.time() - last_push_time > push_cooldown:

                send_notification(distance)

                last_push_time = time.time()

        elif distance < 10:

            color = (255, 0, 0)

        else:

            color = (0, 255, 255)

        movement = classify_movement(
            prev_positions.get(i, y),
            y
        )

        prev_positions[i] = y

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            color,
            2
        )

        cv2.putText(
            frame,
            f"{movement}, {distance:.1f}m",
            (x, max(y - 10, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2
        )

    out.write(frame)

    frame_count += 1

    if frame_count % 100 == 0:
        print("Processed frames:", frame_count)

cap.release()
out.release()

print("Detection completed!")
print("Total frames:", frame_count)

Processed frames: 100
Processed frames: 200
Processed frames: 300
Processed frames: 400
Processed frames: 500
Processed frames: 600
Detection completed!
Total frames: 632


In [9]:
from IPython.display import FileLink

print("Pedestrian detection completed successfully!")
print("Processed video saved as:", output_path)

display(FileLink(output_path))

Pedestrian detection completed successfully!
Processed video saved as: pedestrian_output.mp4


/content/pedestrian_output.mp4

In [10]:
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>